### BulkFormer feature extraction

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  

In [2]:
import math
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr
from collections import OrderedDict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset,DataLoader,random_split
from torch_geometric.typing import SparseTensor

In [3]:
from utils.BulkFormer import BulkFormer

In [4]:
from model.config import model_params

In [5]:
device = 'cuda'

In [6]:
graph_path = 'data/G_gtex.pt'
weights_path = 'data/G_gtex_weight.pt'
gene_emb_path = 'data/esm2_feature_concat.pt'

In [7]:
graph = torch.load(graph_path, map_location='cpu', weights_only=False)
weights = torch.load(weights_path, map_location='cpu', weights_only=False)
graph = SparseTensor(row=graph[1], col=graph[0], value=weights).t().to(device)
gene_emb = torch.load(gene_emb_path, map_location='cpu', weights_only=False)
model_params['graph'] = graph
model_params['gene_emb'] = gene_emb

In [8]:
model = BulkFormer(**model_params).to(device)

In [9]:
ckpt_model = torch.load('model/Bulkformer_ckpt_epoch_29.pt',weights_only=False)

In [10]:
new_state_dict = OrderedDict()
for key, value in ckpt_model.items():
    new_key = key[7:] if key.startswith("module.") else key
    new_state_dict[new_key] = value

In [11]:
model.load_state_dict(new_state_dict)

<All keys matched successfully>

In [12]:
def extract_feature(expr_array, 
                    high_var_gene_idx,
                    feature_type,
                    aggregate_type,
                    device,
                    batch_size,
                    return_expr_value = False,
                    esm2_emb = None,
                    valid_gene_idx = None):

    expr_tensor = torch.tensor(expr_array,dtype=torch.float32,device=device)
    mydataset = TensorDataset(expr_tensor)
    myloader = DataLoader(mydataset, batch_size=batch_size, shuffle=False) 
    model.eval()

    all_emb_list = []
    all_expr_value_list = []


    with torch.no_grad():
        if feature_type == 'transcriptome_level':
            for (X,) in tqdm(myloader, total=len(myloader)):
                X = X.to(device)
                output, emb = model(X, [2])
                all_expr_value_list.append(output.detach().cpu().numpy())
                emb = emb[2].detach().cpu().numpy()
                emb_valid = emb[:,high_var_gene_idx,:]
     
                if aggregate_type == 'max':
                    final_emb =np.max(emb_valid, axis=1)
                elif aggregate_type == 'mean':
                    final_emb =np.mean(emb_valid, axis=1)
                elif aggregate_type == 'median':
                    final_emb =np.median(emb_valid, axis=1)
                elif aggregate_type == 'all':
                    max_emb =np.max(emb_valid, axis=1)
                    mean_emb =np.mean(emb_valid, axis=1)
                    median_emb =np.median(emb_valid, axis=1)
                    final_emb = max_emb+mean_emb+median_emb

                all_emb_list.append(final_emb)
            result_emb = np.vstack(all_emb_list)
            result_emb = torch.tensor(result_emb,device='cpu',dtype=torch.float32)

        elif feature_type == 'gene_level':
            for (X,) in tqdm(myloader, total=len(myloader)):
                X = X.to(device)
                output, emb = model(X, [2])
                emb = emb[2].detach().cpu().numpy()
                emb_valid = emb[:,valid_gene_idx,:]
                all_emb_list.append(emb_valid)
                all_expr_value_list.append(output.detach().cpu().numpy())
            all_emb = np.vstack(all_emb_list)
            all_emb_tensor = torch.tensor(all_emb,device='cpu',dtype=torch.float32)
            esm2_emb_selected = esm2_emb[valid_gene_idx]
            esm2_emb_expanded = esm2_emb_selected.unsqueeze(0).expand(all_emb_tensor.shape[0], -1, -1)  # [B, N, D]
            esm2_emb_expanded = esm2_emb_expanded.to('cpu')

            result_emb = torch.cat([all_emb_tensor, esm2_emb_expanded], dim=-1)
    
    if return_expr_value:
        return np.vstack(all_expr_value_list)
    
    else:
        return result_emb

In [13]:
def main_gene_selection(X_df, gene_list):

    to_fill_columns = list(set(gene_list) - set(X_df.columns))


    padding_df = pd.DataFrame(np.full((X_df.shape[0], len(to_fill_columns)), -10), 
                            columns=to_fill_columns, 
                            index=X_df.index)

    X_df = pd.DataFrame(np.concatenate([df.values for df in [X_df, padding_df]], axis=1), 
                        index=X_df.index, 
                        columns=list(X_df.columns) + list(padding_df.columns))
    X_df = X_df[gene_list]
    
    var = pd.DataFrame(index=X_df.columns)
    var['mask'] = [1 if i in to_fill_columns else 0 for i in list(var.index)]
    return X_df, to_fill_columns,var

In [14]:
# load demo data
demo_df = pd.read_csv('data/demo.csv')

In [15]:
bulkformer_gene_info = pd.read_csv('data/bulkformer_gene_info.csv')
# fix extra row?
bulkformer_gene_info = bulkformer_gene_info[bulkformer_gene_info['ensg_id'] != '35991']

In [16]:
bulkformer_gene_info

,gene_symbol,ensg_id,gene_type,gene_length,ensp_id,protein_length
0,TSPAN6,ENSG00000000003,protein_coding,4530.0,ENSP00000362111,245.0
1,TNMD,ENSG00000000005,protein_coding,1476.0,ENSP00000362122,317.0
2,DPM1,ENSG00000000419,protein_coding,9276.0,ENSP00000360640,295.0
3,SCYL3,ENSG00000000457,protein_coding,6883.0,ENSP00000356744,742.0
4,C1orf112,ENSG00000000460,protein_coding,5970.0,ENSP00000286031,853.0
...,...,...,...,...,...,...
20006,ENSG00000289791,ENSG00000289791,protein_coding,1316.0,ENSP00000476965,288.0
20007,ENSG00000289809,ENSG00000289809,protein_coding,778.0,ENSP00000403093,163.0
20008,ENSG00000290146,ENSG00000290146,protein_coding,1292.0,ENSP00000480420,223.0
20009,ENSG00000290147,ENSG00000290147,protein_coding,2275.0,ENSP00000499539,201.0


In [17]:
bulkformer_gene_list = bulkformer_gene_info['ensg_id'].to_list()

In [18]:
input_df , to_fill_columns, var= main_gene_selection(X_df=demo_df,gene_list=bulkformer_gene_list)

In [19]:
var.reset_index(inplace=True)
valid_gene_idx = list(var[var['mask'] == 0].index)

In [20]:
high_var_gene_idx = torch.load('data/high_var_gene_list.pt',weights_only=False)

In [21]:
# Extract transcritome-level embedding
result = extract_feature(
    expr_array= input_df.values[:16],
    high_var_gene_idx=high_var_gene_idx,
    feature_type='transcriptome_level',
    aggregate_type='max',
    device=device,
    batch_size=4,
    return_expr_value=False,
    esm2_emb=model_params['gene_emb'],
    valid_gene_idx=valid_gene_idx
)

100%|██████████| 4/4 [00:03<00:00,  1.29it/s]


In [22]:
result.shape

torch.Size([16, 640])

In [23]:
# Extract gene-level embedding
result = extract_feature(
    expr_array= input_df.values[:16],
    high_var_gene_idx=high_var_gene_idx,
    feature_type='gene_level',
    aggregate_type='all',
    device=device,
    batch_size=4,
    return_expr_value=False,
    esm2_emb=model_params['gene_emb'],
    valid_gene_idx=valid_gene_idx
)

100%|██████████| 4/4 [00:03<00:00,  1.29it/s]


In [24]:
result.shape

torch.Size([16, 20010, 1920])

In [25]:
# Extract expression values
result = extract_feature(
    expr_array= input_df.values[:16],
    high_var_gene_idx=high_var_gene_idx,
    feature_type='transcriptome_level',
    aggregate_type='all',
    device=device,
    batch_size=4,
    return_expr_value=True,
    esm2_emb=model_params['gene_emb'],
    valid_gene_idx=valid_gene_idx
)

100%|██████████| 4/4 [00:03<00:00,  1.23it/s]


In [26]:
result.shape

(16, 20010)

In [27]:
ucl_gene_data = pd.read_parquet("../UCLThesis/data/BIOAID_UCL_Oxford_361_combined.parquet")

In [28]:
# consider trying to match on ensembl ids instead next.
ucl_gene_list = list(ucl_gene_data.loc[:, "5S_rRNA":].columns)
print(ucl_gene_list[:4])
len(ucl_gene_list)

['5S_rRNA', 'A1BG', 'A1CF', 'A2M']


29652

In [29]:
bulkformer_gene_list = list(bulkformer_gene_info['gene_symbol'])
len(bulkformer_gene_list)

20010

In [30]:
ucl_gene_set = set(ucl_gene_list)
bulkformer_gene_set = set(bulkformer_gene_list)

In [31]:
common_genes = ucl_gene_set.intersection(bulkformer_gene_set)
len(common_genes)

19338

In [32]:
ucl_gene_set.difference(bulkformer_gene_set)

{'FUNDC2P2',
 'NIFKP6',
 'CYCSP12',
 'OR7E36P',
 'FABP5P11',
 'EEDP1',
 'RPL32P21',
 'HADHAP1',
 'UPF3AP2',
 'PTPRVP',
 'SEC22B4P',
 'ADAMTS7P1',
 'HMGN2P30',
 'ARHGEF34P',
 'GAPDHP22',
 'CRTC1P1',
 'TMEM161BP1',
 'CYCSP26',
 'RNA5SP476',
 'AGGF1P8',
 'HNRNPA3P11',
 'ZDHHC4P1',
 'OR4F7P',
 'CRIP1P3',
 'RPS29P6',
 'NOS2P2',
 'LSM3P3',
 'SEC63P1',
 'CRYGGP',
 'H2BC19P',
 'KRT18P50',
 'DUX4L6',
 'RPS11P7',
 'IGLV3-32',
 'IGSF3P2',
 'DUXAP2',
 'RAD17P2',
 'SUCLA2P1',
 'MTND4LP18',
 'RNA5SP136',
 'GOLGA6L5P',
 'BTF3L4P1',
 'KCTD9P4',
 'PRSS3P4',
 'RNA5SP388',
 'DUX4L31',
 'SDAD1P1',
 'SLC9B1P1',
 'RPL31P11',
 'TUBB4BP7',
 'OR5BM1P',
 'HMGB3P27',
 'MTCO3P40',
 'DHFRP1',
 'FAM9CP1',
 'RNA5SP394',
 'RPL15P3',
 'OR7E161P',
 'QRSL1P2',
 'ATG3P1',
 'KIAA0895LP1',
 'EEF1A1P3',
 'IGKV3D-7',
 'OR7A15P',
 'RPS26P43',
 'RNA5SP372',
 'HLA-DPA3',
 'IGLL3P',
 'LINC03006',
 'SLC25A15P2',
 'LAP3P2',
 'TRGJ2',
 'OR7E85BP',
 'SNRPGP16',
 'RPL35P1',
 'RPL39P15',
 'DUX4L7',
 'FTLP16',
 'TPTE2P2',
 'RPL23AP83',

In [33]:
bulkformer_gene_set.difference(ucl_gene_set)

{'ARNTL',
 'ARNTL2',
 'BHLHB9',
 'BTBD11',
 'C10orf99',
 'C11orf53',
 'C16orf72',
 'C17orf64',
 'C19orf71',
 'C7orf61',
 'CBWD1',
 'CBWD2',
 'CBWD3',
 'CBWD5',
 'CBWD6',
 'COLCA2',
 'CYHR1',
 'DDX58',
 'ENSG00000100101',
 'ENSG00000111780',
 'ENSG00000124593',
 'ENSG00000125695',
 'ENSG00000131152',
 'ENSG00000141979',
 'ENSG00000142539',
 'ENSG00000144785',
 'ENSG00000159239',
 'ENSG00000167774',
 'ENSG00000167807',
 'ENSG00000170846',
 'ENSG00000173366',
 'ENSG00000173867',
 'ENSG00000183889',
 'ENSG00000187186',
 'ENSG00000188223',
 'ENSG00000188897',
 'ENSG00000196826',
 'ENSG00000197991',
 'ENSG00000198211',
 'ENSG00000203546',
 'ENSG00000204003',
 'ENSG00000205236',
 'ENSG00000206549',
 'ENSG00000213204',
 'ENSG00000214265',
 'ENSG00000214558',
 'ENSG00000225528',
 'ENSG00000226490',
 'ENSG00000226690',
 'ENSG00000228144',
 'ENSG00000230707',
 'ENSG00000233757',
 'ENSG00000235007',
 'ENSG00000236543',
 'ENSG00000237378',
 'ENSG00000239395',
 'ENSG00000239920',
 'ENSG00000241489',